## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [3]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [4]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 63,555


In [5]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [6]:
documents[1]

Document(metadata={'source': 'knowledge-base/contracts/Contract with United Healthcare Alliance for Healthllm.md', 'doc_type': 'contracts'}, page_content="# Contract with United Healthcare Alliance for Healthllm\n\n**Contract Date:** May 15, 2025\n**Contract Number:** HL-2025-E-0156\n**Parties:**\n- Insurellm, Inc.\n- United Healthcare Alliance, LLC\n\n---\n\n## Terms\n\n1. **Coverage:** Insurellm agrees to provide United Healthcare Alliance with enterprise access to the Healthllm platform, including extensive customization, multi-state support, dedicated infrastructure, and premium support services for their nationwide health insurance operations covering 250,000+ members across 12 states.\n\n2. **Duration:** This agreement is effective for a period of 48 months from the contract date, representing Insurellm's most strategic long-term healthcare partnership.\n\n3. **Payment:** United Healthcare Alliance shall pay custom Enterprise Tier pricing of $52,000 per month for months 1-12, $56

In [18]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# from langchain_text_splitters import CharacterTextSplitter
# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Contract with BrightWay Solutions for Markellm

**Contract Date:** October 5, 2023  
**Contract ID:** INS-2023-0092

### Terms
This contract (“Contract”) is made between Insurellm, a company incorporated in the United States, and BrightWay Solutions, a technology provider specializing in insurance services.

1. **Scope of Services:**  
   Insurellm shall provide BrightWay Solutions access to the Markellm platform under the agreed pricing structure for a duration of one year from the effective date.

2. **Payment Terms:**  
   BrightWay Solutions agrees to pay an initial setup fee of $1,000 for integration services, followed by the Basic Listing Fee of $199 per month for featured listing on Markellm. Payment shall be made within 30 days of invoice.

3. **Service Level Agreement (SLA):**  
   Insurellm commits to a 99.9% uptime for the platform with dedicated support response times not exceeding 4 business hours.' metadata={'source': 

In [20]:
print(f"Second chunk:\n\n{chunks[1]}")

Second chunk:

page_content='3. **Service Level Agreement (SLA):**  
   Insurellm commits to a 99.9% uptime for the platform with dedicated support response times not exceeding 4 business hours.

### Renewal
1. **Automatic Renewal:**  
   This Contract will automatically renew for additional one-year terms unless either party provides a written notice of intent to terminate at least 30 days prior to the renewal date.

2. **Review Period:**  
   Both parties will enter a review period each year, during which they will discuss potential amendments to the pricing or contract terms based on market conditions and performance metrics.

### Features
1. **Access to AI-Powered Matching:**  
   BrightWay Solutions will benefit from the AI algorithms for optimal customer matches, helping them connect with consumers looking for their specific insurance offerings.

2. **Real-Time Quote Availability:**  
   Consumers sourced via BrightWay Solutions will receive real-time quotes, allowing for a seaml

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [27]:
# Pick an embedding model

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 413 documents


In [28]:
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7f9fa6923230>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7f9fa505b1d0>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [29]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 3,072 dimensions in the vector store


### Part C: Visualize!

In [30]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [43]:
print(documents[390])

### 2. Real-Time Eligibility Verification
Integrated eligibility checking ensures instant verification of coverage status, reducing claim denials and improving provider satisfaction. The system connects with healthcare provider networks for seamless real-time validation.

### 3. AI-Driven Claims Adjudication
Automated claims processing uses machine learning to review claims for accuracy, identify potential fraud, and expedite legitimate payments. The system learns from historical data to continuously improve processing accuracy and efficiency.

### 4. Predictive Healthcare Analytics
Advanced analytics identify high-risk members who would benefit from preventive care interventions, enabling insurers to implement proactive wellness programs that reduce long-term costs and improve member health outcomes.


In [44]:
print(documents[8])

4. **Advanced AI Claims Adjudication:** Enterprise-grade claims processing:
   - Auto-adjudication rate of 85%+ for clean claims
   - Real-time fraud detection with 92%+ accuracy
   - Predictive payment integrity scoring
   - Provider billing pattern analysis
   - Upcoding and unbundling detection
   - Coordination of benefits (COB) automation
   - Subrogation opportunity identification

5. **Population Health Management:** Comprehensive analytics for value-based care:
   - Real-time risk stratification across entire member population
   - Chronic disease identification and progression modeling
   - Predictive analytics for 30+ conditions
   - Hospital readmission prevention protocols
   - Care gap analysis for HEDIS and STAR measures
   - Social determinants of health (SDOH) integration
   - Health equity analytics and disparity identification


In [41]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>index: {i}<br>Text: {d[:100]}..." for i, (t, d) in enumerate(zip(doc_types, documents))],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [32]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()